# FastAPI 기반 AI 모델 서빙 교안

## 1. 배경 및 개요

### 1.1 FastAPI와 AI 모델 서빙
- AI 모델을 개발하는 것만으로는 실제 서비스에서 바로 사용할 수 없습니다.  
- 학습된 모델을 웹이나 앱에서 사용할 수 있도록 연결하는 과정이 **모델 서빙(Model Serving)** 입니다.  
- FastAPI는 Python 기반의 고성능 웹 프레임워크로, 간단한 코드만으로 REST API 서버를 만들고, AI 모델을 효율적으로 배포할 수 있습니다.

### 1.2 왜 FastAPI인가?
- 비동기 처리 지원으로 다수 요청 처리 가능
- 자동 문서화(Swagger) 지원으로 API 테스트가 편리
- Python 친화적이며, 기존 AI 라이브러리(PyTorch, TensorFlow)와 호환 용이



### 1.3 시장 수요
- AI 모델을 서비스에 연결하려는 연구자와 개발자 증가
- 모바일, 웹, 클라우드 환경에서 실시간 예측 필요
- 서버 개발 경험이 적은 초심자도 쉽게 배포 가능



### 1.4 학습 목표

- FastAPI로 AI 모델 서빙 RESTful API 설계 및 구현 가능
- Pydantic을 활용한 데이터 검증 및 설정 가능
- Google Cloud, Amazon EC2 등 클라우드 환경에 배포 가능
- HTTP 기반 서버 구조, 요청/응답, 상태 코드 이해
- 이미지, 텍스트 등 다양한 입력 데이터 처리
- POST, GET 등 HTTP Method 활용 API 구현
- Swagger UI를 통해 테스트 및 디버깅 가능

## 2. 모델 서빙 개념

### 2.1 모델 서빙이란?
AI 모델은 **훈련(Training)** 단계를 통해 데이터를 학습합니다.  
하지만 모델이 학습된 상태로만 있으면, 연구자가 직접 코드를 실행해야만 결과를 볼 수 있습니다.  
즉, **사용자(고객)** 입장에서는 쓸 수 없는 상태죠.  

그래서 필요한 것이 바로 **모델 서빙(Model Serving)** 입니다.  
→ 학습된 모델을 **웹 서비스**처럼 배포하여, 누구나 쉽게 **요청(request)을 보내면 결과(response)를 받을 수 있도록 만드는 과정**입니다.

예를 들어:
- 내가 학습한 "고양이 vs 강아지 분류 모델"이 있다고 합시다.
- 연구자는 `predict("dog.jpg")` 같은 코드를 실행해서 결과를 볼 수 있습니다.
- 하지만 일반 사용자는 Python을 모를 수도 있고, 코드를 실행할 수도 없습니다.
- 그래서 이 모델을 **앱이나 웹사이트**에서 쓸 수 있게 만드는 게 바로 **서빙**입니다.

  <img src="image/HTTP.webp" width="500">  

이미지 출처 : https://www.freepik.com/premium-vector/rest-api-model-representational-state-transfer-paradigm-from-client-server_410581593.htm

### 2.2 모델 서빙 방법

AI 모델을 실제 서비스에서 사용하려면, 외부 프로그램이 모델을 호출할 수 있도록 **서빙(Serving)** 해야 한다.  
서빙 방식은 크게 **4가지 방법**으로 분류할 수 있다.

#### 2.2.1 직접 API 서버 개발 (FastAPI, Flask)

- **방법:** Python 웹 프레임워크(FastAPI, Flask)를 사용해 직접 REST API 서버를 만든다.  
- **장점:** 필요한 기능을 자유롭게 커스터마이즈 가능. (전처리/후처리, 인증, 로깅 등)  
- **단점:** 서버 운영, 배포, 확장성, 트래픽 증가 대응 등을 모두 직접 관리해야 한다.

예시:  
사용자가 이미지를 업로드하면 `/predict`로 요청을 보내고,  
서버 내부에서 모델이 추론한 뒤 `"강아지"` 같은 JSON 결과를 반환한다.

<img src="image/fastapi_flow.png" width="600">

#### 2.2.2 웹 기반 서빙(Web UI 방식)

- **방법:** 모델을 웹페이지 형태로 제공하여 사용자가 직접 브라우저에서 모델 기능을 사용할 수 있도록 제공한다.  
  (예: Gradio, Streamlit, HuggingFace Spaces)  
- **장점:** API나 서버 개발 없이도 빠르게 데모와 서비스 화면을 만들 수 있음.  
- **단점:** 고트래픽 서비스에는 부적합하며, UI 중심이라 복잡한 비즈니스 로직 구현은 제한적.

예시: 손글씨 숫자 인식 모델을 Gradio로 만들면, 사용자가 웹페이지에서 이미지를 올리면 바로 예측 결과가 표시됨.

<img src="image/wen.png" width="400">





#### 2.2.3 서빙 전용 라이브러리 사용  
(TensorFlow Serving, TorchServe, Triton Inference Server)

- **방법:** 모델을 전용 서빙 엔진에 등록하면 자동으로 `/predict` API가 생성된다.  
- **장점:**  
  - 고성능 추론(배치 처리, GPU 최적화, 멀티 모델 배포)  
  - 표준화된 API 제공  
  - 운영 환경에 맞춘 최적화 기능 포함

- **단점(유연성이 떨어지는 이유):**  
  - **비즈니스 로직을 자유롭게 넣기 어려움**  
    - 예: 요청 포맷 커스터마이즈, 인증/로그인, 복잡한 후처리 등을 직접 넣기 어려움  
  - **내부 동작 방식이 고정되어 있어 구조 변경이 어려움**  
  - **‘모델 서빙’ 기능에만 초점**  
    → 서비스 로직을 모델 서버 안에 억지로 넣기 어려워 API 서버를 따로 만들어야 하는 경우가 많음  
  - **사용자 입력을 복잡하게 가공해야 하는 서비스에는 부적합**

예시:  
PyTorch 모델을 TorchServe에 등록하면 자동으로 `/predictions/model_name` API가 제공된다.

<img src="image/torchserve.png" width="400">

## 3. 백엔드 프로그래밍 기본

### 3.1 서버 구조 이해
서버(Server)는 **클라이언트(Client)의 요청(Request)을 받아 응답(Response)을 보내는 시스템**입니다.  
클라이언트는 브라우저, 스마트폰 앱, 다른 서버 등이 될 수 있습니다.

#### REST API란?

REST API(Representational State Transfer API)는 웹가 앱이 서로 데이터를 주고받기 위한 대표적인 통신 방식입니다.  
웹에서 **자원(Resource)을 URI**로 표현하고, 그 자원을 **HTTP Method(GET/POST/PUT/DELETE 등)** 로 조작하는 방식입니다.

쉽게 말해:

- **URI = 어떤 자원인지 나타내는 주소**  (예: `/users`, `/items/10`, `/predict`)
- **HTTP Method = 그 자원에 대해 어떤 행동을 할지 나타내는 규칙** (예: 조회, 생성, 수정, 삭제)

REST API는 다음과 같은 원칙을 따릅니다:

1. **자원(Resource)은 URL(URI)로 표현한다.**  
   - `/users` → 사용자라는 자원  
   - `/users/1` → 아이디 1번 사용자

2. **행동(조작)은 HTTP Method로 표현한다.**  
   - `GET /users` → 사용자 목록 조회  
   - `POST /users` → 사용자 생성  
   - `PUT /users/1` → 사용자 정보 전체 수정  
   - `DELETE /users/1` → 사용자 삭제
   - URL에는 GET/POST 같은 단어가 표시되지 않으며, **HTTP Method는 요청 헤더(Request Line)에 포함되어 구분된다.**

3. **Stateless(무상태성)**  
   - 서버는 이전 요청의 상태를 기억하지 않음  
   - 매 요청이 독립적 → 확장성과 속도가 좋아짐

REST API는 현대 웹 서버와 모바일 앱 대부분이 사용하는 가장 표준적인 방식입니다.

#### HTTP Method (REST API에서 사용하는 행동 규칙)

- **GET**: 자원 조회(Read)  
  예: 뉴스 기사 목록 불러오기

- **POST**: 자원 생성(Create)  
  예: 회원가입 정보 저장

- **PUT**: 자원 전체 수정(Update – 전체 교체)  
  예: 프로필 전체 변경

- **PATCH**: 자원 부분 수정(Update – 일부 교체)  
  예: 닉네임만 변경

- **DELETE**: 자원 삭제(Delete)  
  예: 댓글 삭제

#### Request / Response 구조

##### Request (요청: Client → Server)
- **Header**  
  예: Content-Type, Authorization 토큰  
  예시)  
  ```
  GET /users/1 HTTP/1.1
  Host: example.com
  Authorization: Bearer ABCD1234
  Content-Type: application/json
  ```
- **Body**  
  예: 업로드한 이미지, JSON 데이터 등 실제 데이터  
  예시)  
  ```
  {
    "name": "minsu",
    "age": 25
  }
  ```

##### 실제 HTTP Request의 구조 예시
```makefile
POST /users/1 HTTP/1.1
Host: example.com
Authorization: Bearer ABCD1234
Content-Type: application/json
Content-Length: 32

{
  "name": "minsu",
  "age": 25
}
```


##### Response (응답: Server → Client)
- **Status Code**  
  예: 200, 404, 500  
- **Body**  
  예: 예측 결과(JSON), HTML 페이지  
  예시)  
  ```
  HTTP/1.1 200 OK
  Content-Type: application/json

  {
    "user_id": 1,
    "name": "minsu",
    "age": 25
  }
  ```

#### HTTP Status Code (예시 포함)

- **200 OK**  
  - 의미: 요청이 정상적으로 처리됨  
  - 예시:  ```GET /users/1 → 사용자의 정보가 정상적으로 응답됨```

- **400 Bad Request**  
  - 의미: 요청 형식이 잘못됨(필수 데이터 누락, JSON 오류 등)  
  - 예시:  
    ```
    POST /users  (body에 name이 없을 때)
    {"age": 20 }
    → 서버: "name 필드가 없습니다" → 400 오류
    ```

- **401 Unauthorized**  
  - 의미: 인증 토큰 없음 또는 만료  
  - 예시:  
    ```
    GET /mypage  (Authorization 토큰 없음)
    → 로그인 필요 → 401 발생
    ```

- **404 Not Found**  
  - 의미: 요청한 자원이 존재하지 않음  
  - 예시:  
    ```
    GET /users/99999
    → 99999번 사용자 없음 → 404 반환
    ```

- **500 Internal Server Error**  
  - 의미: 서버 내부에서 예기치 않은 오류 발생  
  - 예시:  
    ```
    GET /products
    → DB 연결 오류 발생 → 서버가 500 반환
    ```

### 요청과 응답 예시

**요청(Request)**  
```
POST /predict
Host: example.com
Content-Type: application/json
Content-Length: 63
{
  "image_url": "https://example.com/cat.jpg"
}
```

**응답(Response)**  
```
HTTP/1.1 200 OK
Content-Type: application/json

{
  "result": "cat",
  "confidence": 0.98
}
```

### IP와 Port 개념

- **IP 주소**  
  - 인터넷에서 컴퓨터(또는 스마트폰, 서버)가 어디에 있는지 구분하는 “집 주소” 역할을 합니다.  
  - 예: `192.168.0.1`, `203.0.113.15`  
  - 이 주소 덕분에 우리가 보내는 요청이 정확한 장치로 전달될 수 있습니다.

- **Port 번호**  
  - 같은 집(IP 주소) 안에서도 여러 개의 “방(서비스)”이 존재할 수 있습니다.  
  - Port는 그 방의 번호로, 어떤 서비스로 들어갈지 알려주는 역할을 합니다.  
  - 예:  
    - 80번 → 일반 웹사이트(HTTP)  
    - 443번 → 보안 연결 웹사이트(HTTPS)  
    - 3306번 → MySQL DB  
    - 6379번 → Redis  

- **비유**  
  - **IP = 아파트 주소**  
  - **Port = 그 아파트의 특정 호수**  
  - 예를 들어,  
    - “서울시 강남구 101동(=IP)” 안의  
    - “1502호(=Port)”에 있는 서비스에 방문한다고 생각하면 됩니다.  

- **간단한 실제 예시**  
  - 아래 주소로 접속하면…  
    ```
    http://192.168.0.10:8000
    ```
    - IP = 192.168.0.10  
    - Port = 8000 (FastAPI 개발 서버에서 자주 사용)  
  - → 즉, “192.168.0.10 서버의 8000번 서비스에 접속하겠다”는 뜻입니다.

### 동기 / 비동기 처리

#### 동기(Synchronous)
- 요청을 **하나씩 순서대로 처리**하는 방식  
- 앞선 요청이 끝나야만 다음 요청이 실행됨  
- 단순하지만, 여러 요청이 몰리면 기다림이 길어질 수 있음

- **쉬운 비유**  
  - ATM 기기 1대 앞에서 줄을 서는 상황  
  - 앞사람이 뭔가를 오래 하면 내가 계속 기다려야 함

- **실제 예시**  
  - 서버가 이미지 1장을 처리하는 동안 다음 요청이 대기  
  - 파일 업로드를 동기 처리하면, 파일 하나 전송 끝날 때까지 다음 업로드가 진행되지 않음

#### 비동기(Asynchronous)
- 요청을 **여러 개 동시에 처리**하는 방식  
- 기다리는 동안 다른 요청도 먼저 진행할 수 있어 전체 효율이 높아짐  
- 실시간 서비스에서 매우 유용 (채팅, 알림, 이미지 처리 등)

- **쉬운 비유**  
  - 패스트푸드점 주문 시스템  
  - 주문만 받고 나중에 번호를 불러주는 방식  
  - 손님은 기다리는 동안 다른 일을 할 수 있고, 주방은 동시에 여러 주문을 처리함

- **실제 예시**  
  - API 서버가 동시에 여러 이미지 분석 요청을 처리  
  - 웹사이트가 여러 사용자의 채팅 메시지를 실시간으로 처리  
  - 비동기 방식의 대표 기술: Node.js, Python의 `async/await`

## 4. FastAPI 소개 및 기본 지식

### 4.1 FastAPI 특징
**FastAPI**는 최근 가장 많이 사용되는 Python 웹 프레임워크 중 하나입니다.  

<img src="image/swagger.png" width="700">

- **Python 기반, 고성능**  
  → 비동기(Async) 처리와 함께 매우 빠른 속도를 제공.  
- **자동 문서화 지원**  
  → 코드를 작성하면 Swagger UI, ReDoc을 통해 API 문서를 자동으로 확인 가능.  
- **비동기 처리 지원**  
  → 동시에 여러 요청을 효율적으로 처리할 수 있어 AI 모델 서빙에 적합. 

### 4.2 FastAPI의 기본 문법: 데코레이터, 타입 힌트, 함수 구조

#### ✔ 데코레이터(Decorator)란?
Python에서 함수 위에 `@`를 붙여 함수의 동작을 수정하거나 확장하는 문법입니다.

**기본 형태:**
```python
@app.get("/경로")  # ← 데코레이터
def 함수명(매개변수):
    return 데이터
```

**의미:**
- `@app.get()` = "이 함수를 GET 요청이 들어올 때 실행해라"
- `/경로` = "어떤 URL에서 이 함수를 실행할지"
- `함수` = "실행할 실제 코드"

#### ✔ 실제 예시로 이해하기

```python 
from fastapi import FastAPI

app = FastAPI()

# 데코레이터: "이 함수는 GET /hello 요청을 처리해"
@app.get("/hello")
def hello():
    return {"message": "안녕하세요"}

# 클라이언트가 http://localhost:8000/hello 접속
# → FastAPI가 자동으로 hello() 함수 실행
# → {"message": "안녕하세요"} JSON 반환
```

#### ✔ 데코레이터 종류 (HTTP Method)

| 데코레이터 | 역할 | 예시 |
|-----------|------|------|
| `@app.get()` | 조회(Read) | 사용자 정보 가져오기 |
| `@app.post()` | 생성(Create) | 새 데이터 저장 |
| `@app.put()` | 전체 교체(Update) | 기존 데이터 변경 |
| `@app.delete()` | 삭제(Delete) | 데이터 제거 |

#### ✔ 왜 데코레이터를 사용하나?

**장점:**
- URL과 함수를 자동으로 연결 (라우팅)
- 코드가 간단하고 직관적
- FastAPI가 자동으로 문서화도 생성

#### 타입 힌트(Type Hints) - 함수 매개변수 정의

FastAPI의 강력한 기능은 **Python 타입 힌트**에서 나옵니다.

#### ✔ 타입 힌트란?
함수의 매개변수가 **어떤 데이터 타입인지 명시**하는 문법입니다.

```python
def read_item(item_id: int, q: str = None):
    #              ↑ int 타입   ↑ str 타입
```

**각 부분 설명:**
- `item_id: int` = "item_id는 정수(int) 타입"
- `q: str` = "q는 문자열(str) 타입"
- `= None` = "기본값은 None (없어도 OK)"

#### ✔ 타입 힌트의 효과

**1) 자동 타입 검증**
```python
# 잘못된 요청
GET /items/abc?q=hello
     ↑ 문자열인데 정수 타입 요청
→ FastAPI: "item_id는 정수여야 합니다" (400 오류)
```

**2) 자동 문서 생성**
- Swagger UI에서 "item_id는 정수" 자동 표시
- 사용자가 올바른 형식으로 입력하도록 유도

**3) IDE 자동완성**
- 개발 도구가 타입을 인식해 코드 자동완성 제공

#### ✔ 자주 사용하는 타입들

```python
# 기본 타입
item_id: int           # 정수
price: float           # 실수
name: str              # 문자열
is_active: bool        # 참/거짓

# 선택적 타입 (Python 3.10+)
q: str | None = None   # 문자열 또는 None

# 구 버전 Python
from typing import Optional
q: Optional[str] = None  # 위와 동일
```

#### ✔ 실제 예시

```python
@app.get("/users/{user_id}")
def get_user(user_id: int, limit: int = 10):
    # user_id: 정수 (필수)
    # limit: 정수 (선택, 기본값 10)
    return {"user_id": user_id, "limit": limit}

# 올바른 요청
GET /users/123?limit=20
→ {"user_id": 123, "limit": 20}

# 잘못된 요청
GET /users/abc
→ 422 오류: user_id는 정수여야 합니다
```

#### 함수의 반환값과 JSON

FastAPI는 우리가 반환하는 **Python 객체(딕셔너리, 리스트, Pydantic 모델 등)** 을  
클라이언트에게 전송할 때 **자동으로 JSON 형식으로 변환하여 응답**합니다.  

즉, 개발자는 JSON 문자열을 직접 만들 필요가 없고,  
단순히 Python 객체를 `return`하기만 하면 됩니다.

```python
@app.get("/items/{item_id}")
def read_item(item_id: int, q: str = None):
    # 여기서 item_id와 q는 URL로 전달된 값이며, JSON이 아님
    return {
        "received_id": item_id,
        "search_keyword": q,
        "message": "요청을 잘 받았습니다!"
    }

# 위의 Python 딕셔너리를 FastAPI가 자동으로 JSON으로 변환하고
# Content-Type, 상태 코드 등의 HTTP 응답을 생성하여 클라이언트에 전송함
```

**이 “자동 JSON 변환”이 FastAPI의 대표적인 장점입니다!**

### 4.3 설치 및 기본 구조
FastAPI 설치와 실행은 매우 간단합니다.  

#### 1) FastAPI & Uvicorn 설치
```bash
pip install fastapi uvicorn
```

#### 2) 기본 앱 파일(main.py) 생성
`main.py` 파일 생성 후 아래 코드 삽입
```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"message": "Hello, FastAPI"}

@app.get("/items/{item_id}")
def read_item(item_id: int, q: str | None = None):
    return {"item_id": item_id, "q": q}
```

#### 3) 개발 서버 실행

```bash
uvicorn main:app --reload
```
`--reload` 옵션 → 코드를 수정하면 서버가 자동으로 재시작되어 개발할 때 매우 편리합니다.

### 4.4 FastAPI 서버 테스트 방법

FastAPI는 테스트하는 방법이 다양합니다.  
초심자도 바로 따라 할 수 있도록 3가지 방식을 소개합니다.

#### 방법 1) 웹 브라우저 테스트 (가장 간단)

아래 주소를 브라우저에 입력:

```http://127.0.0.1:8000/```

→ JSON 응답:  ```{"message": "Hello, FastAPI"}```


특정 아이템 조회: ```http://127.0.0.1:8000/items/123?q=hello```

→ JSON 응답:  ```{"item_id":123,"q":"hello"}```

#### 방법 2) curl로 터미널에서 테스트

```bash
curl -s http://127.0.0.1:8000/
```

응답:
```json
{"message": "Hello, FastAPI"}
```

아이템 조회:

```bash
curl -s "http://127.0.0.1:8000/items/999?q=python"
```

In [1]:
!curl -s http://127.0.0.1:8000/

{"message":"Hello, FastAPI"}

#### 방법 3) 자동 문서화된 Swagger UI 사용

FastAPI는 자동으로 API 문서를 생성합니다.

브라우저에서 아래 주소 접속:

```
http://127.0.0.1:8000/docs
```

→ GET/POST 버튼을 눌러 API를 직접 실행해볼 수 있음  
→ Request/Response 구조를 자동으로 보여줘서 매우 편리  
→ 초보자/테스터/기획자도 쉽게 사용 가능

ReDoc 스타일 문서도 제공됨:

```
http://127.0.0.1:8000/redoc
```

#### 요약

| 테스트 방식 | 특징 |
|------------|------------------------------------------------|
| **웹 브라우저** | GET 요청 빠른 확인 |
| **curl** | 터미널에서 간단 자동화 테스트 |
| **Swagger UI (/docs)** | 가장 편리 • 요청/응답 확인 • 파라미터 테스트 가능 |

### 4.5 Path & Query Parameter

이제 FastAPI에서 **실제로 데이터를 받고 보내는 방법**을 배워보겠습니다.  
FastAPI에서 API로 값을 전달하는 주요 방식은 두 가지입니다:

#### ✔ 1) Path Parameter (필수 정보)
URL 경로의 일부가 되어, **어떤 자원(resource)을 요청하는지 지정하는 값**입니다.

예:
```
/items/10
→ item_id = 10번 아이템을 조회하겠다
```

Path Parameter는 **반드시 있어야 하는 값**이며,  
보통 “특정한 하나의 대상”을 찾을 때 사용합니다.  

비유: **아파트 주소(건물동/호수)를 찾는 것 → 꼭 있어야 하는 정보**


#### ✔ 2) Query Parameter (선택 정보)
`?` 뒤에 붙는 값으로, **검색 조건·필터·옵션 같은 추가 정보**입니다.

예:
```
/items/10?q=apple
→ item_id=10번 아이템 중에서 q=apple 조건을 적용
```

Query Parameter는 **선택적(optional)** 이며,  
주로 “추가 조건” 또는 “옵션”을 제공할 때 사용합니다.

비유:
- **택배 요청 시 선택하는 옵션 (문 앞 배송, 경비실 보관 등)**  
- 주소는 고정이지만 옵션은 선택하는 것


#### ✔ 3) 실제 FastAPI 코드 예시

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/items/{item_id}")
def read_item(item_id: int, q: str = None):
    return {"item_id": item_id, "q": q}
```

- `item_id` → **Path Parameter** (필수)  
- `q` → **Query Parameter** (선택)  
  - `None` = 기본값 없음(= 전달 안 해도 OK)


#### 요약

| 구분 | Path Parameter | Query Parameter |
|------|----------------|-----------------|
| 목적 | 어떤 자원을 요청하는지 지정 | 옵션·필터·검색 조건 |
| 필수 여부 | 필수 | 선택 |
| URL 형태 | `/items/10` | `/items/10?q=apple` |
| 비유 | 아파트 주소 | 배달 옵션 |


이렇게 두 파라미터의 **역할**과 **언제 사용하는지**가 정확히 구분되면 헷갈리지 않습니다.

### 4.6 Request/Response Body

웹 API에서는 클라이언트와 서버가 서로 **데이터를 주고받는 통로**가 필요합니다.  
FastAPI에서는 이 데이터를 대부분 **JSON 형식**으로 주고받으며,  
이미지·문서 같은 **파일 업로드**도 쉽게 처리할 수 있습니다.

#### ✔ Request Body (요청 본문)
클라이언트가 서버로 보내는 실제 데이터입니다.

예:
- 회원가입 데이터  
  ```json
  { "name": "홍길동", "age": 25 }
  ```
- 이미지 파일 업로드
- 분석 요청을 위한 텍스트, 옵션 값 등

#### ✔ Response Body (응답 본문)
서버가 처리한 결과를 클라이언트에게 돌려주는 데이터입니다.

예:
```json
{ "status": "success" }
```

URL은 주소(Path/Query), Body는 실제 전달물(내용물)이라고 생각하면 이해가 빠릅니다.

### 4.7 예시: JSON + 파일 업로드 처리

아래 예시는 **JSON 데이터 + 파일 업로드**를 동시에 처리하는 API입니다.  
실제 모델 서빙 시, 사용자의 추가 정보(JSON)와 업로드한 이미지·문서를 함께 처리하는 구조에 자주 사용됩니다.

```python
from fastapi import FastAPI, File, UploadFile
from pydantic import BaseModel

app = FastAPI()

class PredictRequest(BaseModel):
    name: str
    age: int

@app.post("/predict/")
async def predict(request: PredictRequest, file: UploadFile = File(...)):
    return {
        "name": request.name,
        "age": request.age,
        "filename": file.filename
    }
```

- **JSON Body로 받는 데이터:**  
  - `name`, `age`  
  - FastAPI가 `PredictRequest` 모델을 통해 자동 검증 및 타입 변환 처리

- **파일 업로드로 받는 데이터:**  
  - `file: UploadFile`  
  - 이미지, 문서, PDF 등 다양한 파일 형식 지원

#### 이 구조가 왜 중요한가?
- 실전 AI 서비스(예: 얼굴 인식, 제품 사진 분류 등)에서는  
  **“파일(이미지)” + “요청 옵션(JSON)”** 조합이 매우 흔하게 사용됨  
- FastAPI는 이를 매우 간단하게 표현할 수 있어 서버 설계가 깔끔해짐

#### 요약
| 구분 | 의미 | 예시 |
|------|------|------|
| **Request Body** | 서버로 보내는 실제 데이터 | JSON, 파일, 폼 데이터 |
| **Response Body** | 서버가 돌려주는 결과 | JSON, 메시지 |
| **주요 활용** | 모델 입력, 폼 제출, 파일 업로드 | 이미지 + 옵션, 문서 + 파라미터 |

### 문제 1. FastAPI에서 Path Parameter와 Query Parameter의 예를 하나씩 쓰시오.  

<details>
<summary>정답 보기</summary>

Path Parameter 예시
- 정의: URL 경로의 일부  
- 형식: `/items/{item_id}`  
- 요청 예: `/items/10`  
→ Path Parameter = `10`

Query Parameter 예시
- 정의: `?` 뒤에 오는 추가 옵션  
- 형식: `?q=검색어`  
- 요청 예: `?q=apple`  
→ Query Parameter = `q=apple`

</details>


### 문제 2. FastAPI에서 Path Parameter와 Query Parameter의 차이를 간단히 설명하시오.

<details>
<summary>정답 보기</summary>

- **Path Parameter:**  
  URL 경로 자체에 포함되어 “특정 자원”을 지정하는 필수 정보  
  예: `/users/1`

- **Query Parameter:**  
  옵션·필터·검색 조건 등 “추가 정보”를 전달하는 선택적 값  
  예: `/users/1?active=true`

</details>

### 실습 문제 1. 현재 시간을 알려주는 API 만들기 — `/time`

지금까지 배운 내용을 토대로, **아주 간단한 API 하나를 main.py에 직접 추가해보세요.**

**API 기능 요구 사항**  
1. 아래 URL로 요청을 보내면 서버가 아래처럼 응답하도록 만들어보세요
 - 요청 : http://127.0.0.1:8000/time
 - 응답 : {"message": "현재 시간은 2025년 01월 01일 15시 30분 입니다."}  
2. **응답 형식(JSON)** : `{"message": "현재 시간은 XXXX년 XX월 XX일 XX시 XX분 입니다."}`
3. 현재 시간을 “한국 시간(KST)” 기준으로 표현**

힌트: 아래 함수는 “현재 시간을 문자열로 반환하는 함수”입니다.  
**이 함수를 main.py에 그대로 붙여넣어도 됩니다.**

In [ ]:
from datetime import datetime, timedelta, timezone

def get_korean_time_string():
    # 한국 시간대(KST, +9시간)
    KST = timezone(timedelta(hours=9))
    now = datetime.now(KST)

    return f"현재 시간은 {now.year}년 {now.month}월 {now.day}일 {now.hour}시 {now.minute}분 입니다."

#### ✔ 테스트 방법

- 브라우저로 테스트  
  ```
  http://127.0.0.1:8000/time
  ```
- Swagger UI로 테스트  
  ```
  http://127.0.0.1:8000/docs
  ```
  → GET /time 클릭 → Execute

- 터미널로 테스트
  ```
  curl http://localhost:8000/time
  ```

<details>
<summary>정답 코드 펼치기</summary>

```python
# main.py

from fastapi import FastAPI
from datetime import datetime, timedelta, timezone

app = FastAPI()

@app.get("/time")
def get_korean_time_string():
    # 한국 시간대(KST, +9시간)
    KST = timezone(timedelta(hours=9))
    now = datetime.now(KST)
    message = f"현재 시간은 {now.year}년 {now.month}월 {now.day}일 {now.hour}시 {now.minute}분 입니다."
    return {"message": message}
    
```
</details>

### POST 방식 & Request Body 기초 이해하기

**POST 방식과 Request Body(JSON)** 를 다뤄봅니다.  
지금까지는 URL로 전달되는 *Path*와 *Query* 파라미터만 사용했지만,  
서버에 **JSON 데이터를 보내서 처리하려면 POST + Request Body**를 사용해야 합니다.


#### ✔ POST는 왜 필요한가?
- GET은 “데이터 주세요” → Body를 보내지 않음  
- POST는 “데이터 보낼 테니 처리해주세요” → Body(JSON)를 보낼 수 있음  

즉, **서버로 JSON 데이터를 보낼 때는 POST 방식이 자연스럽고 표준적**입니다.

#### ✔ Request Body(JSON)란?
클라이언트가 서버로 보내는 실제 데이터입니다. 아래처럼 생긴 JSON을 서버로 전달합니다.

```json
{
  "message": "hello"
}
```

FastAPI에서는 Request Body를 가장 간단히 다음처럼 받을 수 있습니다:

```python
@app.post("/echo")
async def echo(body: dict):
    message = body.get("message")   # dict에서 원하는 값 꺼내기
    return {"received": message}
```

즉,  
- `body: dict` : JSON이 자동으로 파이썬 딕셔너리로 변환되어 들어옴  
- `body.get("키")` : dict에서 원하는 값 꺼내기  

이 두 가지만 이해하면 **Request Body 기초는 완전히 끝!**


### 실습 2. JSON Body를 받아서 그대로 응답하는 `/echo` API 만들기

아래 조건을 만족하는 새로운 API를 **main.py에 직접 추가**하세요.

#### 요구사항 
1) `POST /echo` 엔드포인트 만들기  
2) JSON Body에 `"message"` 값을 받기  
3) 서버는 `"received"` 키로 다시 돌려주기  

### 예시 입력(JSON Body)
```json
{
  "message": "hello"
}
```

### 예시 출력(JSON Response)
```json
{
  "received": "hello"
}
```

Swagger UI(`http://127.0.0.1:8000/docs`)에서 테스트하면 가장 쉽습니다.


curl POST 요청 : 
    
curl -X POST http://127.0.0.1:8000/echo \
  -H "Content-Type: application/json" \
  -d '{"message":"hello"}'

<details>
<summary>정답 코드 펼치기</summary>

```python
from fastapi import FastAPI, Request

app = FastAPI()

@app.post("/echo")
async def echo(body: dict):
    message = body.get("message")          # dict에서 "message" 값 꺼내기
    return {"received": message}           # JSON Response 자동 변환
```

</details>


### 실습 문제 3. 새로운 API 만들기: `/profile/{user_id}`

아래 조건을 만족하는 **새로운 API 하나를 직접 만들어보세요. (main.py에 추가)**  

**API 기능 요구 사항**
1. GET은 Request Body를 받을 수 없기 때문에, 이 문제는 POST로 구현해야 합니다.
2. **Path Parameter**
- `user_id` (int)
- 예: `/profile/10` → user_id = 10

3. **Query Parameter**
- `verbose` (bool, 선택 / 기본값: false)
- 사용하는 방법:
  - `/profile/10?verbose=true`
  - 또는 생략하면 자동으로 `false` 처리됨  
  - (FastAPI는 `?verbose=true`, `?verbose=false` 를 자동으로 bool로 변환함)

4. **Request Body (JSON)**
아래와 같은 JSON 데이터를 Body로 전달해야 한다.

```json
{
  "name": "홍길동",
  "age": 30
}
```

5. **Response Body (JSON)**
서버는 다음과 같은 JSON 응답을 반환해야 한다.

```json
{
  "user_id": 10,
  "name": "홍길동",
  "age": 30,
  "verbose": true,
  "message": "Profile received"
}
```

<details> <summary> 테스트 방법 안내 (중요!)</summary>

아래는 다양한 테스트 방법을 제공하지만,    **초심자라면 Swagger UI만 사용해도 충분합니다.**  
curl 명령어는 선택 사항이며, 익숙하지 않아도 괜찮아요.

### 1) 브라우저 (GET은 Path 및 Query만 확인 가능)
```
http://127.0.0.1:8000/profile/10?verbose=true
```

### 2) Swagger UI (가장 쉬운 방법 — 추천)
```
http://127.0.0.1:8000/docs
```
- `POST /profile/{user_id}` 클릭  
- `user_id` 입력  
- `verbose` 입력 (true / false 또는 비워두기 가능)
- JSON Body 입력 후 실행(Execute)

---

### 3) curl 테스트 (선택 사항)
초보자는 curl을 몰라도 전혀 문제 없습니다.  
단축 테스트를 원하면 아래 명령어를 실행해보세요.

```bash
curl -X POST "http://127.0.0.1:8000/profile/10?verbose=true" \
    -H "Content-Type: application/json" \
    -d "{\"name\": \"홍길동\", \"age\": 30}"
```

</details>

<details> <summary>정답 보기</summary>

```python
from fastapi import FastAPI, Request

app = FastAPI()

@app.post("/profile/{user_id}")
async def create_profile(user_id: int, body: dict, verbose : bool = False):
    return {
        "user_id" : user_id,
        "name" : body.get("name"),
        "age" : body.get("age"),
        "verbose" : verbose,
        "message" : "Profile received"}
```
</details>

## 5. Pydantic 활용

### 5.1 Pydantic이란?

**Pydantic**은 Python에서 데이터를 **검증(Validation)** 하고 **구조화(Modeling)** 하기 위한 라이브러리입니다.  
FastAPI는 Request Body를 받을 때 이 Pydantic을 활용하여:

- 데이터 유형이 맞는지 검사하고  
- 누락된 필드가 없는지 확인하고  
- 잘못된 입력이 있으면 자동으로 에러를 반환합니다.

#### ✔ 왜 굳이 Pydantic을 배우는가?

FastAPI는 **Pydantic 없이도 작동합니다.**  
예를 들어 다음처럼 `dict`로 요청을 받아도 됩니다:

```python
@app.post("/echo")
async def echo(body: dict):
    return {"received": body.get("message")}
```

하지만 API가 커지면 아래와 같은 문제가 생깁니다:

- 값이 빠져 있어도 오류가 안 뜬다  
- 타입이 잘못 들어와도 오류가 안 뜬다  
- 어떤 구조의 JSON이 들어오는지 문서(스웨거)에 표시되지 않는다  
- 실수한 사용자가 보내는 잘못된 데이터 때문에 서버가 이상하게 동작할 수 있다

그래서 실무에서는 **안전성 + 자동 문서화 + 유지보수성** 때문에 대부분 Pydantic을 사용합니다.

즉,  
> ✔ 동작은 하지만  
> ✔ **안전한 API를 만들고 싶으면 Pydantic이 필요하다**

### 5.2 Pydantic 기본 사용법 (FastAPI 연동)

#### 1) 먼저 데이터 모델(BaseModel) 정의

```python
from pydantic import BaseModel
from typing import Optional

class User(BaseModel):
    username: str
    age: int
    email: Optional[str] = None
```

- `BaseModel` 상속 → Pydantic 모델 정의  
- 타입 힌트 기반으로 자동 검증  
- `Optional[str]` → 값이 없어도 허용

#### 2) FastAPI에서 Request Body에 적용

Pydantic 모델을 함수의 매개변수로 넣으면  
FastAPI가 자동으로 JSON을 파싱하고 검증합니다.

```python
@app.post("/users/")
async def create_user(user: User):
    return {
        "username": user.username,
        "age": user.age,
        "email": user.email
    }
```

#### ✔ 검증 실패 예시

요청:

```json
{
  "username": "홍길동",
  "age": "스물다섯"
}
```

FastAPI가 자동으로 아래와 같은 에러를 반환:

```json
{
  "detail": [
    {
      "loc": ["body", "age"],
      "msg": "value is not a valid integer",
      "type": "type_error.integer"
    }
  ]
}
```

### 5.3 유의점

- Pydantic 검증은 매우 편리하지만 **필드가 많거나 규칙이 복잡하면 성능 저하 가능**  
- 일반적인 웹 서비스에서는 성능 문제 거의 없음  
- 필요한 필드만 포함하여 단순한 데이터 구조를 유지하는 것이 이상적

### 요약

| 항목 | 설명 |
|------|------|
| 동작 자체는 dict도 가능함 | 하지만 검증이 없어서 위험 |
| Pydantic은 자동 검증 제공 | 잘못된 입력을 서버에 전달하지 않음 |
| Swagger 문서 자동 생성 | Request Body 구조가 정확히 표시됨 |
| 실무에서 거의 필수 | 안전성 + 유지보수성 향상 |

Pydantic은 **FastAPI를 FastAPI답게 사용하는 핵심 기능**입니다.

### 복습 문제 Pydantic 사용 시 얻는 장점은 무엇인가요?  
<details> <summary>정답 보기</summary>

자동 데이터 검증과 타입 체크를 통해 서버의 안정성을 높일 수 있다.
</details>

## 6. 보안 설정 - CORS

### 6.1 CORS란?

우리가 웹에서 API를 만들면, 언젠가는 **다른 곳**에서(API 서버 외부에서) 이 API를 호출하게 됩니다.

- 웹 프론트엔드(React, HTML)
- 모바일 앱
- 다른 서버
- 도커 컨테이너
- 같은 컴퓨터지만 포트가 다른 경우

그런데 **브라우저는 기본적으로 “다른 주소(origin)”에서 온 요청을 위험하다고 생각합니다.**  
그래서 다음과 같은 보안 규칙이 있습니다:

> **브라우저는 다른 주소(origin)에서 온 요청을 기본적으로 막는다.**

이 보안 규칙을 “CORS(Cross-Origin Resource Sharing)”이라고 부릅니다.

쉽게 말하면:

> “이 페이지랑 저 API 서버 주소가 다르면 위험할 수 있으니 일단 막아둘게!”

이런 정책이 없다면 악성 사이트가 사용자의 쿠키/토큰틀 자동으로 포함해서 은행, 메일, 내부 서비스 등에 요청을 보낼수가 있습니다

### 6.2 Origin(출처)이란?

브라우저는 아래 세 가지가 모두 같아야 “같은 출처(Same-Origin)”라고 판단합니다:

- 프로토콜: http / https  
- 도메인: localhost / example.com  
- 포트: 3000 / 8000  

이 중 하나라도 다르면 **다른 출처**입니다.

예)

- `http://localhost:8000`  
- `http://localhost:3000` → 포트가 다르니까 다른 출처!

즉, 같은 컴퓨터라도 포트가 다르면 브라우저 입장에서는 “다른 사이트”입니다.

### 6.3 언제 CORS 문제가 생기나요?

✔ **웹 브라우저에서 실행되는 프론트엔드 → API 호출할 때**  
✔ 주소가 서로 다르면 브라우저가 요청을 차단합니다.

예)
```
프론트: http://localhost:3000
백엔드: http://localhost:8000
```

이 경우 API 요청을 보내면 브라우저가 이렇게 말합니다:

> “주소가 다르네? 위험하니까 요청 못 보내!”

그래서 개발할 때 흔히 이런 에러를 보게 됩니다:

```
Blocked by CORS policy
```

### 6.4 언제 CORS 에러가 안 나나요?

생각보다 “CORS가 아예 작동하지 않는 상황”이 꽤 있습니다.

- Python requests  
- curl  
- Postman  
- Swagger UI  
- HTML 파일을 file://로 직접 열었을 때

왜냐하면 **CORS는 “브라우저 보안 규칙”이기 때문**입니다.  
브라우저가 아닐 때는 CORS가 아예 작동하지 않습니다

### 6.5 그럼 결국 CORS는 왜 필요할까요?

우리가 만든 API는 결국 **외부에서 접근**하게 됩니다.

- 프론트엔드 개발 서버  
- 도커 컨테이너  
- 모바일 앱(fetch 기반)  
- 다른 서버에서 호출  
- 실제 배포 환경

이 모든 경우를 생각하면, CORS를 배우는 것이 자연스럽습니다.

즉:

> **API를 외부와 연결할 계획이 있다면 CORS는 언젠가 반드시 배우게 된다.**

### 6.6 FastAPI에서 CORS 해결하기

브라우저는 기본적으로 이렇게 행동합니다:

> "내가 보고 있는 웹 페이지의 주소와  
> API 서버 주소가 다르면 위험할 수 있어!  
> 허락받지 않은 요청이면 막을게!"

즉, 브라우저는 **API 요청을 막기 전에 먼저 백엔드에게 허락 여부를 물어본다.**

그래서 FastAPI(백엔드)가 이렇게 말해줘야 한다:

> "그 요청 허용해도 됩니다(Allow)."

이 “허락장”을 보내는 기능이 바로 **CORS 설정(Cross-Origin Resource Sharing)** 입니다.

#### ✔ FastAPI에서 브라우저에게 "허용합니다"라고 보내는 방법

```python
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],     # 어떤 주소에서 요청해도 허용
    allow_methods=["*"],     # 모든 메소드(GET, POST 등) 허용
    allow_headers=["*"],     # 모든 요청 헤더 허용
)
```

#### ✔ 왜 백엔드에서 CORS를 설정하나요?

여기서 헷갈릴 수 있는 단어가 바로 **프론트엔드**입니다.

일반적으로 프론트엔드를 설명할 때  
“웹 화면, 버튼, 디자인…”을 떠올리지만,  
CORS에서는 **조금 다른 의미**로 사용됩니다.

#### ✔ CORS에서의 역할 설명 

- **백엔드(FastAPI)** = 데이터를 가지고 있는 “서버 주인”
- **브라우저(Chrome, Safari 등)** = 출입문을 지키는 “보안 담당자”
- **프론트엔드(웹 페이지에서 실행되는 JavaScript 코드)** = 서버에 데이터를 요청하는 “손님처럼 행동하는 코드”

즉,

- 프론트엔드 = **사용자가 브라우저에서 여는 웹사이트**  
  (그리고 그 안에서 실행되는 JavaScript 코드가 서버에 요청을 보냄)

#### ✔ 정확한 흐름

1. 사용자가 웹사이트(프론트엔드)를 연다  
2. 그 웹사이트 안의 JavaScript 코드가 FastAPI에 요청을 보낸다  
3. 브라우저가 가로채서 이렇게 묻는다:  
   > “이 웹사이트(origin)에서 이 서버에 접근해도 되나요?”

4. FastAPI가 CORS 설정으로 허락 여부를 브라우저에게 전달한다  
   - 허락 O → 요청 진행  
   - 허락 X → 브라우저가 요청을 차단

#### ✔ 핵심 요약

프론트엔드라고 해서 "사용자"가 직접 데이터를 요청하는 게 아니다.  
**웹 페이지 안에서 실행되는 JavaScript 코드(AJAX, fetch, axios 등)** 가  
서버에 요청을 보내는 것이다.

그리고 브라우저가 이 요청을 검사한다.

그래서 CORS는 이렇게 정리할 수 있다:

> - “요청을 보내는 건 프론트엔드 코드”  
> - “요청을 막을지 말지 결정하는 건 브라우저”  
> - “브라우저에게 허락장을 주는 건 백엔드(FastAPI)”

#### ✔ 모든 Origin 허용은 위험

`allow_origins=["*"]`  
→ “누구나 내 API를 불러도 됩니다!”라는 뜻  

개발 단계에서는 편하지만 실서비스에서는 위험합니다.

안전하게 만들려면 특정 Origin만 허용해야 합니다:

```python
allow_origins=[
    "http://localhost:3000",
    "https://my-frontend.com"
]
```

### 6.7 요약

| 질문 | 답변 |
|------|------|
| CORS란? | 브라우저가 "다른 주소(origin)"에서 온 API 요청을 막는 보안 기능 |
| 왜 필요한가? | 악성 사이트가 몰래 API를 호출하는 것을 막기 위해 |
| 언제 발생하나요? | 프론트엔드 ↔ API 서버 주소(origin)가 다를 때 |
| 왜 FastAPI에서 설정하나요? | 백엔드가 허락장을 보내야 브라우저가 요청을 풀어주기 때문 |
| 개발 단계에서는? | `allow_origins=["*"]` 로 편하게 시작 가능 |
| 배포 단계에서는? | 특정 도메인만 정확히 허용해야 안전 |


### 복습 문제 1. CORS 설정의 목적을 한 문장으로 쓰시오.  
<details> <summary>정답 보기</summary>
외부 도메인에서 오는 HTTP 요청을 허용하거나 차단하기 위해 설정한다.
</details>

## 7. FastAPI 예시 및 연습 문제

#### 예시 1. 회원가입 API (POST /signup)

#### 요구사항
- 사용자가 이름, 이메일, 나이를 보내면 서버는 “회원가입 완료” 메시지를 반환한다.

#### 요청(JSON)
```json
{
  "name": "김철수",
  "email": "test@example.com",
  "age": 25
}
```

#### 응답(JSON)
```json
{
  "message": "회원가입 완료!",
  "user": {
    "name": "김철수",
    "email": "test@example.com",
    "age": 25
  }
}
```

### 예시 코드
```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

# Request Body 데이터 구조 정의 (Pydantic)
class SignupRequest(BaseModel):
    name: str
    email: str
    age: int

@app.post("/signup")
def signup(data: SignupRequest):
    # data는 Pydantic으로 자동 검증된 객체
    return {
        "message": "회원가입 완료!",
        "user": data
    }
```


### 예시 2. BMI 계산 API (POST /bmi)

#### 요구사항
- 키(cm)와 몸무게(kg)를 입력하면 BMI를 계산해줌  
  BMI = 몸무게(kg) / (키(m)^2)

####  요청(JSON)
```json
{
  "height": 170,
  "weight": 65
}
```

#### 응답(JSON)
```json
{
  "height": 170,
  "weight": 65,
  "bmi": 22.49
}
```

####  예시 코드
```python
from pydantic import BaseModel

class BMIRequest(BaseModel):
    height: float  # cm
    weight: float  # kg

@app.post("/bmi")
def calculate_bmi(data: BMIRequest):
    height_m = data.height / 100  # cm → m
    bmi = data.weight / (height_m * height_m)

    return {
        "height": data.height,
        "weight": data.weight,
        "bmi": round(bmi, 2)
    }
```


### 예시 3. 로그인 API (POST /login)

#### 요구사항
- id, password를 입력하면 성공 여부를 반환

####  요청
```json
{
  "id": "admin",
  "password": "1234"
}
```

#### 응답
```json
{
  "login": true,
  "message": "로그인 성공"
}
```

#### 예시 코드
```python
from pydantic import BaseModel

class LoginRequest(BaseModel):
    id: str
    password: str

@app.post("/login")
def login(data: LoginRequest):
    # 실제 서비스에서는 DB 확인 필요
    if data.id == "admin" and data.password == "1234":
        return {"login": True, "message": "로그인 성공"}
    else:
        return {"login": False, "message": "로그인 실패"}
```

### 예시 4. 상품 등록 API (POST /product)

#### 요구사항
- 상품명, 가격, 재고를 입력하면 “등록 완료” 메시지 반환

####  요청
```json
{
  "name": "사과",
  "price": 1200,
  "stock": 50
}
```

#### 응답
```json
{
  "message": "상품 등록 완료",
  "product": {
    "name": "사과",
    "price": 1200,
    "stock": 50
  }
}
```

####  코드
```python
from pydantic import BaseModel

class Product(BaseModel):
    name: str
    price: int
    stock: int

@app.post("/product")
def add_product(data: Product):
    return {
        "message": "상품 등록 완료",
        "product": data
    }
```

### 예시 5. 좌표 거리 계산 API (POST /distance)

#### 요구사항
- 두 좌표 (x1, y1), (x2, y2)를 입력하면 두 점 사이 거리 반환

거리 공식:  
distance = √((x2 - x1)² + (y2 - y1)²)

####  요청
```json
{
  "x1": 0,
  "y1": 0,
  "x2": 3,
  "y2": 4
}
```

#### 응답
```json
{
  "distance": 5.0
}
```

####  코드
```python
from pydantic import BaseModel
import math

class PointDistance(BaseModel):
    x1: float
    y1: float
    x2: float
    y2: float

@app.post("/distance")
def calculate_distance(data: PointDistance):
    dx = data.x2 - data.x1
    dy = data.y2 - data.y1
    dist = math.sqrt(dx*dx + dy*dy)
    return {"distance": dist}
```

### 연습문제 (쉬움)

정답 코드는 pydantic을 사용하여 작성되었습니다.  
참고해주세요~

### 문제 1. 인사 메시지 만들기 API

**요구사항**  
- POST `/greet`  
- Request Body(JSON):  
  - name: 문자열  
- 응답 예시  
  ```json
  { "message": "안녕하세요, 홍길동님!" }
  ```
- pydantic 사용

<details><summary>정답</summary>

```python
from pydantic import BaseModel

class GreetRequest(BaseModel):
    name: str

@app.post("/greet")
def greet(request: GreetRequest):
    return {"message": f"안녕하세요, {request.name}님!"}
```

</details>

### 문제 2. 두 숫자의 합과 평균 API

**요구사항**  
- POST `/math`  
- Request Body(JSON):  
  - a: 정수  
  - b: 정수  
- 응답 예시  
  ```json
  { "sum": 30, "avg": 15.0 }
  ```

<details><summary>정답</summary>

```python
from pydantic import BaseModel

class MathRequest(BaseModel):
    a: int
    b: int

@app.post("/math")
def math(request: MathRequest):
    s = request.a + request.b
    return {"sum": s, "avg": s / 2}
```

</details>



### 문제 3. 나이를 받아 성인 여부 판별 API

**요구사항**  
- POST `/check-age`  
- Request Body(JSON):  
  - age: 정수  
- 응답 예시  
  ```json
  { "is_adult": true }
  ```

<details><summary>정답</summary>

```python
from pydantic import BaseModel

class AgeRequest(BaseModel):
    age: int

@app.post("/check-age")
def check_age(request: AgeRequest):
    return {"is_adult": request.age >= 19}
```

</details>

### 문제 4. 상품 가격 계산 API

**요구사항**  
- POST `/product`  
- Request Body(JSON):  
  - name: 문자열  
  - price: 정수  
  - count: 정수  
- 응답 예시  
  ```json
  { "name": "사과", "total": 12000 }
  ```

<details><summary>정답</summary>

```python
from pydantic import BaseModel

class ProductRequest(BaseModel):
    name: str
    price: int
    count: int

@app.post("/product")
def product(req: ProductRequest):
    return {"name": req.name, "total": req.price * req.count}
```

</details>

### 문제 5. 문자열 분석 API

**요구사항**  
- POST `/analyze`  
- Request Body(JSON):  
  - text: 문자열  
- 응답 예시  
  ```json
  { "length": 17 }
  ```

<details><summary>정답</summary>

```python
from pydantic import BaseModel

class AnalyzeRequest(BaseModel):
    text: str

@app.post("/analyze")
def analyze(request: AnalyzeRequest):
    return {"length": len(request.text)}
```

</details>

### 중급 예시

중급 예시는 실제 API 구조에 자주 등장하는 패턴을 포함합니다.  
각 코드에는 충분한 주석이 달려 있어, 문제를 풀기 위한 참고 자료 역할을 합니다.

### "회원 가입 정보"를 받는 API (email은 선택값)
Optional + 기본값 예시  

```python
from pydantic import BaseModel
from typing import Optional

class Signup(BaseModel):
    username: str
    age: int
    email: Optional[str] = None   # 없어도 OK
    is_active: bool = True        # 기본값(True)

@app.post("/signup")
def signup(user: Signup):
    return {
        "username": user.username,
        "age": user.age,
        "email": user.email,
        "is_active": user.is_active
    }
```

### 나이와 점수를 검증하는 API  
Pydantic 유효성 검증 (ge, le 사용)

- 나이(age)는 0 이상  
- 점수(score)는 0~100 사이

```python
from pydantic import BaseModel, Field

class ScoreRequest(BaseModel):
    age: int = Field(..., ge=0)        # age >= 0
    score: int = Field(..., ge=0, le=100)  # 0 <= score <= 100

@app.post("/validate-score")
def validate(req: ScoreRequest):
    return {"status": "ok"}
```

### 배송 정보(주소 + 사용자) 받기
중첩 모델(Nested Model)


```python
from pydantic import BaseModel

class Address(BaseModel):
    city: str
    zipcode: str

class User(BaseModel):
    name: str
    address: Address    # 다른 모델을 포함

@app.post("/order")
def order(user: User):
    return {
        "name": user.name,
        "city": user.address.city,
        "zipcode": user.address.zipcode
    }
```



### 여러 상품 가격 합산 API  
리스트(List) 구조 받기  

JSON 배열을 받아서 총합 계산

```python
from pydantic import BaseModel
from typing import List

class Item(BaseModel):
    name: str
    price: int

class Cart(BaseModel):
    items: List[Item]    # 여러 상품을 받음

@app.post("/cart/total")
def cart_total(cart: Cart):
    total = sum(item.price for item in cart.items)
    return {"total": total}
```

### 특정 user_id의 활동 기록을 저장하는 API
Path + Query + Body 조합


- Path: user_id  
- Query: detailed (True/False)  
- Body: 활동명, 지속시간

```python
from pydantic import BaseModel

class Activity(BaseModel):
    name: str
    minutes: int

@app.post("/users/{user_id}/activity")
def add_activity(user_id: int, activity: Activity, detailed: bool = False):
    return {
        "user_id": user_id,
        "activity": activity.name,
        "minutes": activity.minutes,
        "detailed": detailed
    }
```

### 연습문제(중간)

### 문제 1. Optional 필드 포함한 회원 등록 API 만들기

### 요구사항  
- POST `/register`  
- Request Body(JSON)  
  - username: 문자열  
  - age: 정수  
  - email: 문자열 (선택, 없으면 None)  
  - marketing: bool (선택, 기본값 False)  
- 응답: 입력된 모든 값을 그대로 JSON으로 반환

### 추가 설명  
- email이 없어도 API가 정상 작동해야 함  
- marketing 값이 없으면 False 처리되어야 함

<details><summary>정답 보기</summary>

```python
from pydantic import BaseModel
from typing import Optional

class Register(BaseModel):
    username: str
    age: int
    email: Optional[str] = None
    marketing: bool = False

@app.post("/register")
def register(user: Register):
    return user.dict()
```

</details>

### 문제 2. 숫자 검증이 포함된 API 만들기

### 요구사항  
- POST `/score/check`  
- Request Body(JSON)  
  - math: 정수 (0~100)  
  - english: 정수 (0~100)  
- 응답  
  ```json
  { "total": 합계 }
  ```

### 조건  
Pydantic Field를 활용하여 0~100 범위를 벗어나면 오류 반환되도록 해야 함.

<details><summary>정답 보기</summary>

```python
from pydantic import BaseModel, Field

class Score(BaseModel):
    math: int = Field(..., ge=0, le=100)
    english: int = Field(..., ge=0, le=100)

@app.post("/score/check")
def check(score: Score):
    total = score.math + score.english
    return {"total": total}
```

</details>

### 문제 3. 중첩 모델을 이용한 주문 생성 API

### 요구사항  
- POST `/order/create`  
- Request Body(JSON)
  ```json
  {
    "customer": {"name": "홍길동", "phone": "010-1234-5678"},
    "address": {"city": "서울", "zipcode": "12345"}
  }
  ```
- 응답: 이름과 도시를 묶어 다음 JSON으로 응답
  ```json
  {
    "summary": "홍길동님, 서울로 배송됩니다."
  }
  ```

### 조건  
- customer와 address 두 개의 모델을 만들 것  
- Order 모델을 만들어 두 모델을 포함시키기

<details><summary>정답 보기</summary>

```python
from pydantic import BaseModel

class Customer(BaseModel):
    name: str
    phone: str

class Address(BaseModel):
    city: str
    zipcode: str

class Order(BaseModel):
    customer: Customer
    address: Address

@app.post("/order/create")
def create_order(order: Order):
    msg = f"{order.customer.name}님, {order.address.city}로 배송됩니다."
    return {"summary": msg}
```

</details>



### 문제 4. 여러 상품을 받아 총 금액과 개수를 계산하는 API

### 요구사항  
- POST `/items/summary`  
- Request Body(JSON)
  ```json
  {
    "items": [
      {"name": "사과", "price": 3000},
      {"name": "바나나", "price": 2000}
    ]
  }
  ```
- 응답
  ```json
  {
    "total_price": 5000,
    "count": 2
  }
  ```

### 조건  
- List[Item] 형태로 모델 구성  
- sum()과 len()을 활용

<details><summary>정답 보기</summary>

```python
from pydantic import BaseModel
from typing import List

class Item(BaseModel):
    name: str
    price: int

class ItemList(BaseModel):
    items: List[Item]

@app.post("/items/summary")
def items_summary(data: ItemList):
    total = sum(item.price for item in data.items)
    count = len(data.items)
    return {"total_price": total, "count": count}
```

</details>

### 문제 5. Path + Query + Body 조합 API 만들기

### 요구사항  
- POST `/users/{user_id}/log`  
- Path Parameter  
  - user_id (int)  
- Query Parameter  
  - detail (bool, 기본값 False)  
- Request Body(JSON)  
  - action: 문자열  
  - duration: 정수  
- detail이 true면 `"detail": "상세 로그 저장됨"` 추가  
- detail이 false면 `"detail": "간단 로그 저장됨"` 추가

### 응답 예시  
```json
{
  "user_id": 1,
  "action": "운동",
  "duration": 30,
  "detail": "상세 로그 저장됨"
}
```

<details><summary>정답 보기</summary>

```python
from pydantic import BaseModel

class Log(BaseModel):
    action: str
    duration: int

@app.post("/users/{user_id}/log")
def log_activity(user_id: int, log: Log, detail: bool = False):
    msg = "상세 로그 저장됨" if detail else "간단 로그 저장됨"
    return {
        "user_id": user_id,
        "action": log.action,
        "duration": log.duration,
        "detail": msg
    }
```

</details>

### 어려운 예시 5개

### 예시 1. 이미지 파일 업로드 + JSON Body 동시에 받기  
(파일과 JSON을 함께 처리하는 고급 패턴)

```python
from fastapi import FastAPI, UploadFile, File, Form
from pydantic import BaseModel

app = FastAPI()

class Meta(BaseModel):
    description: str
    category: str

@app.post("/upload")
async def upload_image(
    meta: str = Form(...),   # 문자열로 JSON 받기
    file: UploadFile = File(...)
):
    # meta(JSON 문자열)를 다시 Pydantic 모델로 파싱
    meta_data = Meta.parse_raw(meta)

    return {
        "filename": file.filename,
        "type": file.content_type,
        "description": meta_data.description,
        "category": meta_data.category
    }
```

✔ 특징  
- Form + File + JSON 혼합 처리  
- JSON 문자열을 Pydantic으로 다시 파싱  
- 실제 업로드 기능 구현 가능

#### 참고 : curl 테스트 명령어

```bash
curl -X POST http://localhost:8000/upload \
  -F 'meta={"description":"귀여운 강아지","category":"animal"}' \
  -F 'file=@image/dog.jpg'
```

### 예시 2. 여러 파일 업로드 + 파일 크기 검증

```python
from fastapi import UploadFile, File, HTTPException

@app.post("/multi-upload")
async def multi_upload(files: list[UploadFile] = File(...)):
    if len(files) > 5:
        raise HTTPException(status_code=400, detail="파일은 최대 5개까지만 가능합니다.")

    names = [f.filename for f in files]
    return {"uploaded_files": names}
```

✔ 특징  
- 리스트 형태의 파일 받기  
- 조건 불만족 시 HTTPException 발생  
- 실제 서비스에서 매우 자주 쓰임

### 예시 3. Pydantic 모델 + 검증 + 리스트 + 중첩 종합 패턴

```python
from pydantic import BaseModel, Field
from typing import List

class OrderItem(BaseModel):
    name: str
    price: int = Field(..., ge=0)
    count: int = Field(..., ge=1)

class Order(BaseModel):
    user: str
    items: List[OrderItem]

@app.post("/orders/calc")
def calc_order(order: Order):
    total = sum(item.price * item.count for item in order.items)
    return {
        "user": order.user,
        "items": len(order.items),
        "total_price": total
    }
```

✔ 특징  
- 중첩 모델(Order → OrderItem)  
- 검증 포함(price >= 0, count >= 1)  
- 리스트 합산 로직  
- 실제 쇼핑몰 API 구조

### 예시 4. Enum(열거형)으로 제한된 값만 받기

```python
from enum import Enum

class Role(str, Enum):
    admin = "admin"
    user = "user"
    guest = "guest"

class User(BaseModel):
    name: str
    role: Role   # admin/user/guest 중 하나만 허용

@app.post("/roles")
def role_test(u: User):
    return {"message": f"{u.name}님의 권한은 {u.role} 입니다."}
```

✔ 특징  
- 잘못된 값(예: "superman")이 들어오면 자동으로 422 오류  
- admin/user/guest 같은 고정 값 관리에 매우 좋음

### 예시 5. 요청 Body가 모델 여러 종류 중 하나(Union)일 때 처리

```python
from pydantic import BaseModel
from typing import Union

class TextMessage(BaseModel):
    type: str = "text"
    text: str

class ImageMessage(BaseModel):
    type: str = "image"
    url: str
    width: int
    height: int

Message = Union[TextMessage, ImageMessage]

@app.post("/message")
def send_message(msg: Message):
    # 타입에 따라 분기 처리
    if msg.type == "text":
        return {"received": f"텍스트: {msg.text}"}
    else:
        return {
            "received": "이미지",
            "size": f"{msg.width}x{msg.height}"
        }
```

✔ 특징  
- 서로 다른 형태의 Request Body를 한 엔드포인트에서 처리  
- Union을 활용한 고급 패턴  
- 챗봇/메신저 API에서 자주 사용

### 연습 문제(어려움)

### 문제 1. 상품 이미지 업로드 + JSON 정보 처리 API

### 요구사항  
- POST `/products/upload`
- Form + File 형태로 요청받기
- **Form 데이터(JSON 문자열)**  
  ```json
  {
    "name": "사과",
    "price": 5000,
    "tags": ["과일", "식품"]
  }
  ```
- **파일:** 이미지 1개 (jpg/png 등)
- 응답 예시  
  ```json
  {
    "saved": "사과",
    "filename": "apple.png",
    "tag_count": 2
  }
  ```

### 조건  
- form으로 넘어오는 JSON 문자열을 **Pydantic 모델로 파싱**
- 파일 이름, 상품명, 태그 개수를 반환

<details><summary>정답 보기</summary>

```python
from fastapi import FastAPI, UploadFile, File, Form
from pydantic import BaseModel
import json

app = FastAPI()

class Product(BaseModel):
    name: str
    price: int
    tags: list[str]

@app.post("/products/upload")
async def upload_product(data: str = Form(...), file: UploadFile = File(...)):
    product = Product.parse_raw(data)
    return {
        "saved": product.name,
        "filename": file.filename,
        "tag_count": len(product.tags)
    }
```

</details>

### 문제 2. 주문(Order) 검증 + 예외 처리 API

### 요구사항  
- POST `/order/submit`
- Request Body(JSON)
  ```json
  {
    "items": [
      {"name": "콜라", "price": 1500, "count": 2},
      {"name": "치킨", "price": 18000, "count": 1}
    ]
  }
  ```
- 총 금액이 **2만원 이상이면 주문 가능**  
- 2만원 미만이면 HTTP 400 오류  
  ```json
  {
    "detail": "최소 주문 금액은 20000원입니다."
  }
  ```

<details><summary>정답 보기</summary>

```python
from fastapi import HTTPException
from pydantic import BaseModel, Field
from typing import List

class Item(BaseModel):
    name: str
    price: int = Field(..., ge=0)
    count: int = Field(..., ge=1)

class Order(BaseModel):
    items: List[Item]

@app.post("/order/submit")
def submit_order(order: Order):
    total = sum(i.price * i.count for i in order.items)
    if total < 20000:
        raise HTTPException(status_code=400, detail="최소 주문 금액은 20000원입니다.")
    return {"total": total, "status": "주문 접수 완료"}
```

</details>

### 문제 3. 여러 종류 메시지 처리 (Union 활용)

### 요구사항  
한 엔드포인트에서 **텍스트 메시지** 또는 **이미지 메시지** 둘 다 받을 수 있어야 한다.

- POST `/messages`
- TextMessage 예시  
  ```json
  { "type": "text", "content": "안녕하세요" }
  ```
- ImageMessage 예시  
  ```json
  { "type": "image", "url": "http://...", "width": 300, "height": 200 }
  ```
- 응답: type에 따라 다른 형태로 반환

<details><summary>정답 보기</summary>

```python
from pydantic import BaseModel
from typing import Union

class TextMessage(BaseModel):
    type: str = "text"
    content: str

class ImageMessage(BaseModel):
    type: str = "image"
    url: str
    width: int
    height: int

Message = Union[TextMessage, ImageMessage]

@app.post("/messages")
def messages(msg: Message):
    if msg.type == "text":
        return {"received_text": msg.content}
    else:
        return {"received_image": msg.url, "size": f"{msg.width}x{msg.height}"}
```

</details>

### 문제 4. Enum을 이용한 사용자 권한 설정 API

### 요구사항  
- POST `/auth/role`
- Request Body  
  ```json
  { "username": "kim", "role": "admin" }
  ```
- role은 반드시 admin/user/guest 중 하나  
- role이 잘못되면 자동으로 422 오류 발생  
- 응답:  
  ```json
  { "message": "kim님의 권한은 admin 입니다." }
  ```

<details><summary>정답 보기</summary>

```python
from enum import Enum
from pydantic import BaseModel

class Role(str, Enum):
    admin = "admin"
    user = "user"
    guest = "guest"

class Auth(BaseModel):
    username: str
    role: Role

@app.post("/auth/role")
def set_role(auth: Auth):
    return {"message": f"{auth.username}님의 권한은 {auth.role} 입니다."}
```

</details>

### 문제 5. 유저 정보 업데이트 (Patch 스타일) — 부분 업데이트 처리

### 요구사항  
- PATCH `/users/{user_id}`
- 아래 JSON 중 일부만 들어와도 업데이트 가능해야 함  
  ```json
  {
    "name": "홍길동",
    "email": "test@test.com",
    "age": 30
  }
  ```
- 예: 이름만 바꿀 수도 있음  
  ```json
  { "name": "철수" }
  ```
- 응답: 전달된 값만 업데이트된 형태로 반환

<details><summary>정답 보기</summary>

```python
from fastapi import Path
from pydantic import BaseModel
from typing import Optional

class UserUpdate(BaseModel):
    name: Optional[str] = None
    email: Optional[str] = None
    age: Optional[int] = None

fake_db = {
    1: {"name": "홍길동", "email": "hong@test.com", "age": 25}
}

@app.patch("/users/{user_id}")
def update_user(user_id: int = Path(...), data: UserUpdate = None):
    if user_id not in fake_db:
        return {"error": "존재하지 않는 사용자"}

    original = fake_db[user_id]
    updated = data.dict(exclude_unset=True)  # 전달된 항목만 업데이트
    original.update(updated)

    return original
```

</details>


## 8. 배포 및 운영

### 8.1 클라우드 배포

AI 모델이나 웹 서비스를 실제 사용자에게 제공하려면,  **클라우드 서버**에 배포해야 합니다.  

대표적인 클라우드 서비스:
- **Google Cloud Platform (GCP)** → 무료 크레딧 제공  
- **Amazon AWS (EC2 프리티어)** → 가장 대중적으로 많이 쓰이는 클라우드 서비스

#### EC2에서 FastAPI 실행 예시
1. EC2 인스턴스 생성 (Ubuntu 권장)  
2. FastAPI 및 Uvicorn 설치  
   ```bash
   pip install fastapi uvicorn
   ```
3. 서버 실행  
   ```bash
   uvicorn main:app --host 0.0.0.0 --port 8000
   ```
   - `--host 0.0.0.0`: 외부에서도 접속 가능하도록 설정  
   - `--port 8000`: 서비스 포트 지정  

4. 보안 그룹(Security Group)에서 **포트 8000 열기**  
   → 외부 사용자가 `http://서버IP:8000` 으로 접속 가능

### 8.2 운영 고려사항

서비스를 단순히 실행하는 것만으로는 충분하지 않습니다.  
**운영 단계에서는 안정성과 보안**이 중요합니다.  

- **로그 모니터링**  
  → 에러, 요청 기록을 확인하여 문제 조기 발견  

- **서버 리소스 체크**  
  → CPU, 메모리, 디스크 사용량 모니터링 (예: htop, CloudWatch)  

- **Rate Limiting (요청 제한)**  
  → 특정 사용자가 과도하게 요청을 보내는 것을 방지 (DDOS 완화)  

- **보안 패치 및 업데이트**  
  → OS, Python 패키지를 주기적으로 업데이트하여 취약점 제거  

## 9. 실습 예제: 이미지 분류 모델 서빙

### 9.1 준비
- PyTorch 또는 ONNX 모델 준비
- FastAPI, Uvicorn 설치

```bash
pip install fastapi uvicorn torch torchvision
```

### 9.2 FastAPI 서버 코드 예시
```python
# 아래 예시는 **PyTorch의 사전 학습된 ResNet18 모델**을 FastAPI로 서빙하는 코드입니다.  
from fastapi import FastAPI, File, UploadFile
from torchvision import models, transforms
from PIL import Image
import torch
import io

app = FastAPI()

# 사전 학습된 ResNet18 모델 로드
model = models.resnet18(pretrained=True)
model.eval()

# 이미지 전처리 함수
def transform_image(image_bytes):
    my_transforms = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    image = Image.open(io.BytesIO(image_bytes))
    return my_transforms(image).unsqueeze(0)

# 추론 함수
def get_prediction(image_bytes):
    tensor = transform_image(image_bytes)
    outputs = model(tensor)
    _, y_hat = outputs.max(1)
    return y_hat.item()

# FastAPI 엔드포인트
@app.post("/predict/")
async def predict(file: UploadFile = File(...)):
    image_bytes = await file.read()
    class_id = get_prediction(image_bytes)
    return {"class_id": class_id}
```

#### 코드 설명
- **`UploadFile` + `File(...)`**: 클라이언트에서 업로드한 이미지 파일 처리  
- **전처리(`transform_image`)**: ResNet18이 학습된 입력 크기에 맞게 변환  
- **추론(`get_prediction`)**: 모델로부터 예측 class ID 반환  
- **엔드포인트(`/predict/`)**: 이미지 업로드 → 결과 반환(JSON)

### 9.3 테스트

FastAPI 서버 실행 후, 브라우저에서 Swagger UI에 접속합니다.  

```bash
uvicorn main:app --reload
```

- 접속 주소: [http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs)  
- `/predict/` 엔드포인트 선택  
- 이미지 파일 업로드 → **예측 결과(class_id)** 확인  

문제 9. FastAPI에서 이미지 파일을 POST 방식으로 처리하려면 어떤 클래스를 사용하나요?  

<details> <summary>정답 보기</summary>
```python
# 정답
UploadFile 클래스와 File(...) 선언을 사용한다.
```
</details>

### 복습문제
**문제 10:** FastAPI에서 이미지 파일을 POST 방식으로 처리하려면 어떤 클래스를 사용하나요?  
**정답:** `UploadFile` 클래스와 `File(...)` 선언

## 10. 클라우드 배포 실습

본인이 생각하는 간단한 서비스를 fastapi를 이용하여 클라우드에 배포해 봅시다.

### 10.1 EC2 배포 예시
1. EC2 인스턴스 생성 (Ubuntu)
2. Python, pip, Uvicorn 설치
3. FastAPI 코드 업로드
4. 서버 실행: `uvicorn main:app --host 0.0.0.0 --port 8000`
5. Security Group에서 8000 포트 개방
6. 브라우저에서 Swagger UI 접속

### 10.2 Google Cloud 배포
- Cloud VM 인스턴스 생성
- SSH 접속 후 서버 실행
- 외부 IP로 API 호출 가능

### 복습문제
**문제 13:** EC2에 FastAPI 앱을 배포할 때 외부에서 접근하려면 무엇을 설정해야 하나요?  
**정답:** Security Group에서 FastAPI가 사용하는 포트(예: 8000)를 개방해야 한다.